### Setup - Imports and configuration

In [2]:
import sys
print(sys.executable)

/root/bt4301-group10/BT4301-Project-Group-10/venv/bin/python


In [3]:
import pandas as pd
import numpy as np
import logging
import os
from pathlib import Path
from sqlalchemy import create_engine, text
import mysql.connector

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "dataops":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data"
if not DATA_PATH.exists():
    print(f"Missing {DATA_PATH}. Place datasets there.")

MYSQL_CONFIG = {
    "host": "localhost",
    "user": "bt4301",
    "password": "password",
    "database": "home_credit",
    "port": 3306,
}

In [4]:
# Configure logging for data quality failures
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("home_credit_dq")

#### Load CSV tables from Home Credit

In [5]:
# Load datasets
for f in ["application_train.csv", "application_test.csv", "bureau.csv", "bureau_balance.csv"]:
    p = DATA_PATH / f
    assert p.exists(), f"Missing {f}. Download and place in {DATA_PATH}"
application_train = pd.read_csv(DATA_PATH / "application_train.csv")
application_test = pd.read_csv(DATA_PATH / "application_test.csv")
bureau = pd.read_csv(DATA_PATH / "bureau.csv")
bureau_balance = pd.read_csv(DATA_PATH / "bureau_balance.csv")

print("application_train:", application_train.shape)
print("application_test:", application_test.shape)
print("bureau:", bureau.shape)
print("bureau_balance:", bureau_balance.shape)
application_train.head(2)

application_train: (307511, 122)
application_test: (48744, 121)
bureau: (1716428, 17)
bureau_balance: (27299925, 3)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


#### Data quality checks

In [34]:
def check_missing_values(df: pd.DataFrame, table_name: str, threshold_pct: float = 50.0) -> pd.DataFrame:
    """Report missing values and log columns exceeding threshold."""
    missing = df.isnull().sum()
    pct = (missing / len(df) * 100).round(2)
    report = pd.DataFrame({"missing_count": missing, "missing_pct": pct}).sort_values("missing_pct", ascending=False)
    report = report[report["missing_count"] > 0]
    for col in report[report["missing_pct"] >= threshold_pct].index:
        logger.warning(f"[{table_name}] High missing: {col} = {report.loc[col, 'missing_pct']:.1f}%")
    return report

def check_invalid_categories(df: pd.DataFrame, table_name: str, rules: dict) -> dict:
    """Check categorical columns against allowed values; log invalid."""
    failures = {}
    for col, allowed in rules.items():
        if col not in df.columns:
            continue
        valid = df[col].isin(allowed) | df[col].isna()
        invalid_count = (~valid).sum()
        if invalid_count > 0:
            failures[col] = invalid_count
            logger.warning(f"[{table_name}] Invalid categories in {col}: {invalid_count} rows. Allowed: {allowed}")
    return failures

def check_outliers_iqr(df: pd.DataFrame, table_name: str, numeric_cols: list = None, multiplier: float = 1.5) -> dict:
    """Flag numeric columns with outliers (IQR method) and log summary."""
    if numeric_cols is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    outlier_counts = {}
    for col in numeric_cols:
        if col not in df.columns or df[col].isna().all():
            continue
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        if IQR == 0:
            continue
        low, high = Q1 - multiplier * IQR, Q3 + multiplier * IQR
        n_out = ((df[col] < low) | (df[col] > high)).sum()
        if n_out > 0:
            outlier_counts[col] = int(n_out)
    if outlier_counts:
        logger.info(f"[{table_name}] Outlier counts (IQR): {outlier_counts}")
    return outlier_counts

In [35]:
# Define expected categories for key categorical columns 
APP_CATEGORY_RULES = {
    "NAME_CONTRACT_TYPE": ["Cash loans", "Revolving loans"],
    "CODE_GENDER": ["M", "F", "XNA"],
    "FLAG_OWN_CAR": ["Y", "N"],
    "FLAG_OWN_REALTY": ["Y", "N"],
    "NAME_TYPE_SUITE": ["Unaccompanied", "Family", "Spouse, partner", "Children", "Other_B", "Other_A", "Group of people"],
    "NAME_INCOME_TYPE": ["Working", "Commercial associate", "Pensioner", "State servant", "Unemployed", "Student", "Businessman", "Maternity leave"],
    "NAME_EDUCATION_TYPE": ["Secondary / secondary special", "Higher education", "Incomplete higher", "Lower secondary", "Academic degree"],
    "NAME_FAMILY_STATUS": ["Single / not married", "Married", "Civil marriage", "Separated", "Widow", "Unknown"],
    "NAME_HOUSING_TYPE": ["House / apartment", "With parents", "Municipal apartment", "Rented apartment", "Office apartment", "Co-op apartment"],
    "OCCUPATION_TYPE": [],  
    "WEEKDAY_APPR_PROCESS_START": ["MONDAY", "TUESDAY", "WEDNESDAY", "THURSDAY", "FRIDAY", "SATURDAY", "SUNDAY"],
    "ORGANIZATION_TYPE": [], 
}
BUREAU_CATEGORY_RULES = {
    "CREDIT_ACTIVE": ["Active", "Closed", "Sold", "Bad debt"],
    "CREDIT_CURRENCY": ["currency 1", "currency 2", "currency 3", "currency 4"],
}
BUREAU_BALANCE_RULES = {"STATUS": ["C", "X", "0", "1", "2", "3", "4", "5"]}  # C=closed, X=unknown, 0-5 DPD

In [36]:
# Run data quality checks and log results
dq_results = {}

for name, df, cat_rules in [
    ("application_train", application_train, APP_CATEGORY_RULES),
    ("application_test", application_test, APP_CATEGORY_RULES),
    ("bureau", bureau, BUREAU_CATEGORY_RULES),
    ("bureau_balance", bureau_balance, BUREAU_BALANCE_RULES),
]:
    logger.info(f"--- DQ checks: {name} ---")
    miss = check_missing_values(df, name)
    dq_results[name] = {"missing": miss}
    if not miss.empty:
        display(miss.head(10))
    inv = check_invalid_categories(df, name, {k: v for k, v in cat_rules.items() if v})
    dq_results[name]["invalid_categories"] = inv
    out = check_outliers_iqr(df, name)
    dq_results[name]["outliers"] = out

2026-03-14 04:38:56 [INFO] --- DQ checks: application_train ---
2026-03-14 04:38:56 [WARNING] [application_train] High missing: COMMONAREA_MEDI = 69.9%
2026-03-14 04:38:56 [WARNING] [application_train] High missing: COMMONAREA_AVG = 69.9%
2026-03-14 04:38:56 [WARNING] [application_train] High missing: COMMONAREA_MODE = 69.9%
2026-03-14 04:38:56 [WARNING] [application_train] High missing: NONLIVINGAPARTMENTS_MODE = 69.4%
2026-03-14 04:38:56 [WARNING] [application_train] High missing: NONLIVINGAPARTMENTS_AVG = 69.4%
2026-03-14 04:38:56 [WARNING] [application_train] High missing: NONLIVINGAPARTMENTS_MEDI = 69.4%
2026-03-14 04:38:56 [WARNING] [application_train] High missing: FONDKAPREMONT_MODE = 68.4%
2026-03-14 04:38:56 [WARNING] [application_train] High missing: LIVINGAPARTMENTS_MODE = 68.3%
2026-03-14 04:38:56 [WARNING] [application_train] High missing: LIVINGAPARTMENTS_AVG = 68.3%
2026-03-14 04:38:56 [WARNING] [application_train] High missing: LIVINGAPARTMENTS_MEDI = 68.3%
2026-03-14 

,missing_count,missing_pct
COMMONAREA_MEDI,214865,69.87
COMMONAREA_AVG,214865,69.87
COMMONAREA_MODE,214865,69.87
NONLIVINGAPARTMENTS_MODE,213514,69.43
NONLIVINGAPARTMENTS_AVG,213514,69.43
NONLIVINGAPARTMENTS_MEDI,213514,69.43
FONDKAPREMONT_MODE,210295,68.39
LIVINGAPARTMENTS_MODE,210199,68.35
LIVINGAPARTMENTS_AVG,210199,68.35
LIVINGAPARTMENTS_MEDI,210199,68.35


2026-03-14 04:38:57 [INFO] [application_train] Outlier counts (IQR): {'CNT_CHILDREN': 4272, 'AMT_INCOME_TOTAL': 14035, 'AMT_CREDIT': 6562, 'AMT_ANNUITY': 7504, 'AMT_GOODS_PRICE': 14728, 'REGION_POPULATION_RELATIVE': 8412, 'DAYS_EMPLOYED': 72217, 'DAYS_REGISTRATION': 659, 'OWN_CAR_AGE': 4932, 'CNT_FAM_MEMBERS': 4007, 'HOUR_APPR_PROCESS_START': 2257, 'APARTMENTS_AVG': 10655, 'BASEMENTAREA_AVG': 7190, 'YEARS_BEGINEXPLUATATION_AVG': 4784, 'YEARS_BUILD_AVG': 2154, 'COMMONAREA_AVG': 7942, 'ELEVATORS_AVG': 10420, 'ENTRANCES_AVG': 3882, 'FLOORSMAX_AVG': 5215, 'FLOORSMIN_AVG': 343, 'LANDAREA_AVG': 6888, 'LIVINGAPARTMENTS_AVG': 7881, 'LIVINGAREA_AVG': 12517, 'NONLIVINGAPARTMENTS_AVG': 15580, 'NONLIVINGAREA_AVG': 16546, 'APARTMENTS_MODE': 10397, 'BASEMENTAREA_MODE': 6910, 'YEARS_BEGINEXPLUATATION_MODE': 5074, 'YEARS_BUILD_MODE': 2537, 'COMMONAREA_MODE': 7938, 'ELEVATORS_MODE': 9732, 'ENTRANCES_MODE': 3840, 'FLOORSMAX_MODE': 5104, 'FLOORSMIN_MODE': 320, 'LANDAREA_MODE': 7033, 'LIVINGAPARTMENTS_MOD

,missing_count,missing_pct
AMT_ANNUITY,1226791,71.47
AMT_CREDIT_MAX_OVERDUE,1124488,65.51
DAYS_ENDDATE_FACT,633653,36.92
AMT_CREDIT_SUM_LIMIT,591780,34.48
AMT_CREDIT_SUM_DEBT,257669,15.01
DAYS_CREDIT_ENDDATE,105553,6.15
AMT_CREDIT_SUM,13,0.00


2026-03-14 04:38:58 [INFO] [bureau] Outlier counts (IQR): {'DAYS_CREDIT_ENDDATE': 79340, 'DAYS_ENDDATE_FACT': 1, 'AMT_CREDIT_SUM': 187998, 'AMT_CREDIT_SUM_DEBT': 280455, 'DAYS_CREDIT_UPDATE': 63755, 'AMT_ANNUITY': 43918}
2026-03-14 04:38:58 [INFO] --- DQ checks: bureau_balance ---


#### Cleaning and transformation 

In [37]:
# Clean application_train: fill critical missing with sentinel / mode where appropriate
app_clean = application_train.copy()
# Keep CODE_GENDER missing/XNA as dedicated category "XNA" 
if "CODE_GENDER" in app_clean.columns:
    app_clean["CODE_GENDER"] = app_clean["CODE_GENDER"].fillna("XNA")
# DAYS_EMPLOYED sentinel 365243 = unemployed/pensioner, replace with 0 and add flag for ML
if "DAYS_EMPLOYED" in app_clean.columns:
    app_clean["FLAG_UNEMPLOYED"] = (app_clean["DAYS_EMPLOYED"] == 365243).astype(int)
    app_clean["DAYS_EMPLOYED"] = app_clean["DAYS_EMPLOYED"].replace(365243, 0)
# Cap key amount columns at 99th percentile (train); save caps for application_test (no leakage)
AMT_CAP_COLS = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY"]
AMT_Q99_CAPS = {}
for col in AMT_CAP_COLS:
    if col in app_clean.columns and app_clean[col].notna().any():
        AMT_Q99_CAPS[col] = app_clean[col].quantile(0.99)
        app_clean[col] = app_clean[col].clip(upper=AMT_Q99_CAPS[col])
# Ensure unique SK_ID_CURR (drop duplicates if any, keep first occurrence)
before_dedup = len(app_clean)
app_clean = app_clean.drop_duplicates(subset=["SK_ID_CURR"], keep="first")
if len(app_clean) < before_dedup:
    logger.warning(f"Dropped {before_dedup - len(app_clean)} duplicate SK_ID_CURR rows in application_train.")
# Fill numeric missing with 0 except TARGET 
num_cols = [c for c in app_clean.select_dtypes(include=[np.number]).columns if c != "TARGET" and app_clean[c].isna().any()]
app_clean = app_clean.fillna({c: 0 for c in num_cols})
print("application_train cleaned shape:", app_clean.shape)
app_clean.head(2)

application_train cleaned shape: (307511, 123)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,FLAG_UNEMPLOYED
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [38]:
app_test_clean = application_test.copy()
if "CODE_GENDER" in app_test_clean.columns:
    app_test_clean["CODE_GENDER"] = app_test_clean["CODE_GENDER"].fillna("XNA")
if "DAYS_EMPLOYED" in app_test_clean.columns:
    app_test_clean["FLAG_UNEMPLOYED"] = (app_test_clean["DAYS_EMPLOYED"] == 365243).astype(int)
    app_test_clean["DAYS_EMPLOYED"] = app_test_clean["DAYS_EMPLOYED"].replace(365243, 0)
for col, cap in AMT_Q99_CAPS.items():
    if col in app_test_clean.columns:
        app_test_clean[col] = app_test_clean[col].clip(upper=cap)
before_dedup = len(app_test_clean)
app_test_clean = app_test_clean.drop_duplicates(subset=["SK_ID_CURR"], keep="first")
if len(app_test_clean) < before_dedup:
    logger.warning(f"Dropped {before_dedup - len(app_test_clean)} duplicate SK_ID_CURR rows in application_test.")
num_cols_test = [c for c in app_test_clean.select_dtypes(include=[np.number]).columns if app_test_clean[c].isna().any()]
app_test_clean = app_test_clean.fillna({c: 0 for c in num_cols_test})
print("application_test cleaned shape:", app_test_clean.shape)
app_test_clean.head(2)

application_test cleaned shape: (48744, 122)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,FLAG_UNEMPLOYED
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0,0


In [39]:
bureau_clean = bureau.copy()
bureau_clean = bureau_clean.fillna(0) 
bureau_balance_clean = bureau_balance.copy()
# STATUS: ensure only valid codes
valid_status = ["C", "X", "0", "1", "2", "3", "4", "5"]
if "STATUS" in bureau_balance_clean.columns:
    bureau_balance_clean["STATUS"] = bureau_balance_clean["STATUS"].where(
        bureau_balance_clean["STATUS"].isin(valid_status), "X"
    )
print("bureau_clean:", bureau_clean.shape, "bureau_balance_clean:", bureau_balance_clean.shape)

bureau_clean: (1716428, 17) bureau_balance_clean: (27299925, 3)


#### Load into MySQL


In [40]:
# Create database if it does not exist
conn_root = mysql.connector.connect(
    host=MYSQL_CONFIG["host"],
    user=MYSQL_CONFIG["user"],
    password=MYSQL_CONFIG["password"],
    port=MYSQL_CONFIG["port"],
)
cursor = conn_root.cursor()
cursor.execute(f"CREATE DATABASE IF NOT EXISTS `{MYSQL_CONFIG['database']}`;")
conn_root.commit()
conn_root.close()
print(f"Database `{MYSQL_CONFIG['database']}` ready.")

Database `home_credit` ready.


In [41]:
connection_string = (
    f"mysql+pymysql://{MYSQL_CONFIG['user']}:{MYSQL_CONFIG['password']}"
    f"@{MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}/{MYSQL_CONFIG['database']}"
)
engine = create_engine(connection_string, echo=False)

In [42]:
# Drop existing tables for clean reload 
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS bureau_balance;"))
    conn.execute(text("DROP TABLE IF EXISTS bureau;"))
    conn.execute(text("DROP TABLE IF EXISTS application_test;"))
    conn.execute(text("DROP TABLE IF EXISTS application_train;"))
    conn.commit()

In [43]:
# Load cleaned tables into MySQL 
app_clean.to_sql(name="application_train", con=engine, if_exists="replace", index=False, chunksize=10_000)
app_test_clean.to_sql(name="application_test", con=engine, if_exists="replace", index=False, chunksize=10_000)
bureau_clean.to_sql(name="bureau", con=engine, if_exists="replace", index=False, chunksize=10_000)
bureau_balance_clean.to_sql(name="bureau_balance", con=engine, if_exists="replace", index=False, chunksize=10_000)
print("Loaded application_train, application_test, bureau, bureau_balance into MySQL.")

Loaded application_train, application_test, bureau, bureau_balance into MySQL.


In [ ]:
# Add primary keys and foreign keys for referential integrity (run once and skip if keys exist)
try:
    db = mysql.connector.connect(**MYSQL_CONFIG)
    cursor = db.cursor()
    cursor.execute("ALTER TABLE application_train ADD PRIMARY KEY (SK_ID_CURR);")
    cursor.execute("ALTER TABLE application_test ADD PRIMARY KEY (SK_ID_CURR);")
    cursor.execute("ALTER TABLE bureau ADD PRIMARY KEY (SK_ID_BUREAU);")
    cursor.execute("ALTER TABLE bureau_balance ADD PRIMARY KEY (SK_ID_BUREAU, MONTHS_BALANCE);")
    ##cursor.execute("ALTER TABLE bureau ADD FOREIGN KEY (SK_ID_CURR) REFERENCES application_train(SK_ID_CURR);")
    ##cursor.execute("ALTER TABLE bureau_balance ADDFOREIGN KEY (SK_ID_BUREAU) REFERENCES bureau(SK_ID_BUREAU);")
    db.commit()
    db.close()
    print("Primary and foreign keys added.")
except Exception as e:
    print("Keys may already exist or check schema:", e)

Keys may already exist or check schema: 1068 (42000): Multiple primary key defined
